# Mixed Precision Training

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 06.05 |
| Time | ~70 min |
| Colab GPU | Needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/06_mixed_precision_training.ipynb)

---

## 🎯 Learning Objective

Learn loss scaling and master-weight strategies that make float16/bfloat16 training numerically safe.

---

## 📐 Theory

`04.08` introduced the static loss-scaling trick: multiply the loss by a fixed large constant
before backpropagating, so small gradients don't underflow in float16 (recall `06.01`), then
divide the resulting gradients back down before the optimizer step. This notebook covers what
a *real* mixed-precision training loop does differently: the scale factor isn't fixed, it's
**dynamically adjusted** throughout training, and weights are kept in a separate, higher-precision
**master copy**.

### Why a fixed scale isn't good enough

A fixed scale faces a dilemma: too small, and gradients still underflow to zero in float16
(defeating the purpose); too large, and gradients that were previously fine now *overflow* to
`inf` instead. Since gradient magnitudes shift substantially over the course of training (large
early on, often much smaller as the model converges, and occasionally spiking on an unusual
batch), no single fixed scale works well for an entire run.

### Dynamic loss scaling

**Dynamic loss scaling** treats the scale factor as a moving target, adjusted based on
observed behavior:

1. Start with a large scale (e.g. $2^{16}$).
2. After each backward pass, check whether any gradient is `inf` or `NaN` (an **overflow**).
3. If an overflow occurred: **skip this optimizer step entirely** (the gradients are unusable)
   and **halve the scale** — it was too aggressive for the gradients just seen.
4. If no overflow occurred for `growth_interval` consecutive steps: **multiply the scale by a
   growth factor** (e.g. double it) — training has been stable, so try being more aggressive
   to reduce underflow risk.

This turns loss scaling from a single fixed hyperparameter you'd have to hand-tune into a
self-correcting feedback loop: the scale naturally settles near the largest value that doesn't
cause frequent overflows, without ever needing to know that value in advance.

### Master weights

Even with dynamic scaling, some information genuinely can't survive a round-trip through
float16's precision (recall `06.01`'s machine epsilon discussion) — particularly small weight
updates accumulating on top of much larger existing weights, a scenario prone to the same
catastrophic-cancellation-adjacent precision loss covered in `06.02`. **Mixed precision
training** keeps a **master copy of the weights in float32**, used for the actual optimizer
update; the float16/bfloat16 copies used for the fast forward/backward computation are
refreshed from that float32 master after every step. Compute happens fast, in low precision;
the accumulated *state* (the weights themselves, which must remember many small updates over
time) stays in high precision.

### Putting it together

A full mixed-precision training step looks like: (1) cast weights to float16/bfloat16, (2)
forward and backward pass in low precision, scaling the loss up first, (3) check for
overflow — if found, skip the step and shrink the scale, (4) otherwise, unscale the gradients,
apply the optimizer step to the float32 master weights, and periodically grow the scale back up.
None of this changes *what* is being computed mathematically — it's pure engineering on top of
`04.01`-`04.09`'s optimization theory, exactly as `04.08` noted for its own simpler tricks.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Tracking a dynamic loss scaler's scale factor across training steps shows it steadily growing
while gradients stay well-behaved, then dropping sharply the instant an overflow is detected,
before resuming its climb — a live feedback loop, not a fixed number.

In [ ]:
class DynamicLossScaler:
    def __init__(self, init_scale=2.0**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=3):
        self.scale = init_scale
        self.growth_factor, self.backoff_factor, self.growth_interval = growth_factor, backoff_factor, growth_interval
        self.steps_since_overflow = 0

    def update(self, overflowed):
        if overflowed:
            self.scale *= self.backoff_factor
            self.steps_since_overflow = 0
        else:
            self.steps_since_overflow += 1
            if self.steps_since_overflow >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_overflow = 0

torch.manual_seed(0)
model_viz = torch.nn.Linear(4, 1).double()
scaler = DynamicLossScaler(growth_interval=3)
scale_history, overflow_steps = [], []

for step in range(24):
    x = torch.randn(4, dtype=torch.float64) * (1000 if step in (10, 18) else 1)  # two induced spikes
    model_viz.zero_grad()
    loss = (model_viz(x) ** 2).sum() * 1e-6
    (loss * scaler.scale).backward()
    overflowed = any(torch.isinf(p.grad.half()).any() for p in model_viz.parameters())
    scaler.update(overflowed)
    scale_history.append(scaler.scale)
    if overflowed:
        overflow_steps.append(step)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(scale_history, 'o-', color="#4C72B0", markersize=4)
for s in overflow_steps:
    ax.axvline(s, color="#C44E52", ls="--", alpha=0.7)
ax.set_xlabel("training step"); ax.set_ylabel("loss scale factor (log scale)")
ax.set_title("Dynamic loss scaling: grows steadily, then\nhalves sharply right at each detected overflow (dashed lines)")
plt.show()

print(f"Overflow detected at steps: {overflow_steps}")
print(f"Scale grew from {scale_history[0]:.0f} to {scale_history[-1]:.0f} over {len(scale_history)} steps,")
print(f"dipping at each overflow before resuming growth -- a self-correcting feedback loop, not a fixed number.")

## 🐍 Implementation from Scratch

We implement the full dynamic-loss-scaling training loop -- scale, check for overflow, skip
or step, unscale, periodically grow -- and confirm a master-weights setup keeps float32
precision on the accumulated weights while computation happens in bfloat16.

In [ ]:
class DynamicLossScaler:
    """From-scratch version of torch.cuda.amp.GradScaler's core logic: grow the scale when
    training is stable, shrink it immediately on overflow, and always skip the optimizer
    step for any batch whose gradients actually overflowed."""
    def __init__(self, init_scale=2.0**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=3):
        self.scale = init_scale
        self.growth_factor, self.backoff_factor, self.growth_interval = growth_factor, backoff_factor, growth_interval
        self.steps_since_overflow = 0

    def check_overflow(self, model):
        return any(torch.isinf(p.grad).any() or torch.isnan(p.grad).any()
                   for p in model.parameters() if p.grad is not None)

    def update(self, overflowed):
        if overflowed:
            self.scale *= self.backoff_factor
            self.steps_since_overflow = 0
        else:
            self.steps_since_overflow += 1
            if self.steps_since_overflow >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_overflow = 0

def mixed_precision_step(model_master, model_low_precision, x, y, scaler, opt):
    """One full mixed-precision training step: sync low-precision copy from the float32
    master, compute in low precision, scale/check/unscale, then update the MASTER weights."""
    with torch.no_grad():
        for p_low, p_master in zip(model_low_precision.parameters(), model_master.parameters()):
            p_low.copy_(p_master.to(p_low.dtype))

    for p in model_low_precision.parameters():
        if p.grad is not None:
            p.grad = None
    loss = torch.nn.functional.mse_loss(model_low_precision(x.to(torch.bfloat16)), y.to(torch.bfloat16))
    (loss.float() * scaler.scale).backward()

    overflowed = scaler.check_overflow(model_low_precision)
    if not overflowed:
        opt.zero_grad()
        for p_low, p_master in zip(model_low_precision.parameters(), model_master.parameters()):
            p_master.grad = (p_low.grad.float() / scaler.scale)  # unscale, cast up to float32
        opt.step()  # updates the MASTER (float32) weights, not the low-precision copy
    scaler.update(overflowed)
    return loss.item(), overflowed

torch.manual_seed(2)
model_master = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1))
model_low = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1)).to(torch.bfloat16)
opt = torch.optim.SGD(model_master.parameters(), lr=0.05)
scaler2 = DynamicLossScaler(growth_interval=5)

rng = np.random.default_rng(0)
X = torch.tensor(rng.uniform(-2, 2, size=(50, 4)), dtype=torch.float32)
y = torch.sin(X.sum(dim=1, keepdim=True))

losses = []
for step in range(60):
    loss_val, overflowed = mixed_precision_step(model_master, model_low, X, y, scaler2, opt)
    losses.append(loss_val)
print(f"Mixed-precision training loop: loss {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"Final scale factor: {scaler2.scale:.0f}")

# --- Master weights: why float32 accumulation matters ---
w_master_fp32 = torch.tensor([1000.0], dtype=torch.float32)
w_direct_bf16 = torch.tensor([1000.0], dtype=torch.bfloat16)
small_update, n_updates = 0.01, 500
for _ in range(n_updates):
    w_master_fp32 -= small_update
    w_direct_bf16 -= torch.tensor(small_update, dtype=torch.bfloat16)
true_value = 1000.0 - small_update * n_updates
print(f"\nAfter {n_updates} updates of -{small_update} starting from 1000.0 (true result: {true_value}):")
print(f"  float32 master weight:        {w_master_fp32.item():.4f}")
print(f"  direct bfloat16 accumulation: {w_direct_bf16.item():.4f}  <- every update was silently lost")

## 🤖 Why This Matters for AI

`torch.cuda.amp.GradScaler` (and its more modern `torch.amp.GradScaler` equivalent) implements
exactly the dynamic loss-scaling algorithm derived above, and every major LLM training codebase
(Megatron-LM, DeepSpeed, HuggingFace's Trainer) either uses it directly or reimplements the same
grow/backoff logic. The reason it's ubiquitous rather than optional: without it, engineers would
otherwise need to hand-tune a fixed scale for every new model size, batch size, and learning
rate, and that scale would likely need re-tuning mid-training as gradient statistics shift. Below
we confirm the pieces above compose into a stable training loop, and time-travel back to
`06.01`'s bfloat16-vs-float32 precision discussion to show concretely why master weights in
float32 matter for anything that accumulates many small updates over a long training run.

In [ ]:
import torch.amp as amp

# Confirm PyTorch's own GradScaler exists and follows the SAME grow/backoff pattern as ours
# (enabled=False since this machine is CPU-only; on a CUDA GPU this would actively scale losses)
official_scaler = amp.GradScaler('cuda', enabled=False)
print(f"torch.amp.GradScaler starting scale: {official_scaler.get_scale()}")
print("On a CUDA GPU, GradScaler.scale(loss) / step(opt) / update() implements exactly the")
print("grow-on-success, backoff-on-overflow logic our DynamicLossScaler class reproduces above.")

# Re-run our full mixed-precision loop with a LARGER, longer-running model to make the
# master-weights precision benefit unambiguous over a longer horizon
torch.manual_seed(3)
model_master2 = torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
model_low2 = torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1)).to(torch.bfloat16)
opt2 = torch.optim.SGD(model_master2.parameters(), lr=0.05)
scaler3 = DynamicLossScaler(growth_interval=5)

losses2 = []
for step in range(150):
    loss_val, _ = mixed_precision_step(model_master2, model_low2, X, y, scaler3, opt2)
    losses2.append(loss_val)
print(f"\n150-step mixed-precision run: loss {losses2[0]:.4f} -> {losses2[-1]:.4f}")
print(f"final scale factor reached: {scaler3.scale:.0f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(losses2, color="#4C72B0", lw=2)
ax.set_xlabel("training step"); ax.set_ylabel("loss")
ax.set_title("Full mixed-precision training loop:\nbfloat16 compute, float32 master weights, dynamic scaling")
plt.show()

## 🏋️ Exercises

### Warm-up
1. Trace through `DynamicLossScaler` by hand for 6 steps starting at `scale=1024`,
   `growth_factor=2`, `backoff_factor=0.5`, `growth_interval=2`, with overflows occurring at
   steps 3 and 5 (0-indexed). Write out the scale after each step, then verify with the class.

### Practice
2. Change `growth_interval` in the Visual Intuition cell from 3 to 10, and re-plot. How does a
   longer growth interval change how quickly the scale recovers after an overflow? Explain the
   tradeoff between a short growth interval (recovers scale quickly, but risks more frequent
   overflow-triggered wasted steps) and a long one.

### Challenge
3. In the master-weights comparison, reduce `small_update` from `0.01` to `0.001` and increase
   `n_updates` to `5000` (same total intended change). Does the bfloat16 accumulation error get
   worse, better, or the same relative to the float32 master's error? Explain your answer in
   terms of bfloat16's machine epsilon (recall `06.01`) relative to the current weight magnitude.

---

## 📚 Further Reading
- Micikevicius et al., ["Mixed Precision Training"](https://arxiv.org/abs/1710.03740)
- PyTorch documentation, [Automatic Mixed Precision](https://pytorch.org/docs/stable/amp.html)
- NVIDIA, [Training With Mixed Precision (developer guide)](https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/index.html)

---

*Next notebook: [Numerical Issues in Transformers](07_numerical_issues_in_transformers.ipynb)*